# Paper authors — `author_list`, `team_size`, first and last author

`paper_metadata.parquet` carries an `author_list` column that is **entirely null**: its notebook
ships with `RUN_AUTHORS = False` because filling it means scanning a 40.8 GB gzip, and the column
was left as a schema placeholder. `paper_team_size.parquet` does hold real team sizes for 251.6M
papers — but **nothing in this project writes it.** No notebook, no script, no `WROTE` line in any
job log; only `validation/paper_validation.ipynb` reads it. Its reproduction path is gone.

This notebook rebuilds that information from source, with the author ids and the author order
that `paper_team_size` never carried.

## Source
```
/project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet/works_au_affs_fixed.csv.gz
```
40.8 GB gzip, 15 columns. Three are used: `work_id`, `author_id`, `author_position_int`.

`works_authorships.csv.gz` looks like the natural source and is **not usable**: it is 50 MB — a
truncated extract — and its `author_position` column is empty, so it cannot order authors.

## One row per author is wrong — the file is author × AFFILIATION

This is the trap, and it is large. A row exists for every (work, author, affiliation) triple, so
an author with three affiliations appears three times. Measured on a 300,000-row sample:

| rows per (work, author) | pairs |
|---|---|
| 1 | 211,442 |
| 2 | 24,728 |
| 3 | 6,031 |
| 4 | 2,401 |
| 5+ | 1,398 |

**17.8% of rows are duplicate (work, author) pairs**, and taking the file at face value inflates
mean team size from **2.976 to 3.622 — 22% too high.** The disabled code in `paper_metadata.ipynb`
appends every row without de-duplicating, so had it ever been switched on it would have produced
exactly this inflation. Everything below de-duplicates on `(work_id, author_id)` first, keeping the
author's earliest `author_position_int`.

## Ordering

`author_position` is blank throughout the file; `author_position_int` is the one that carries the
order (1, 2, 3, …) and was non-null in every sampled row. `author_list` is ordered by it, and

- `first_author` = the author at the lowest position
- `last_author` = the author at the highest position

For a single-author paper the two are the same author, by construction rather than by accident.

## Output
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_author.parquet`

| column | |
|---|---|
| `work_id` | OpenAlex id, `W…`, prefix stripped |
| `author_list` | `;`-joined author ids in author order, **de-duplicated** |
| `team_size` | distinct authors — `len(author_list.split(';'))` by construction |
| `first_author` | author id at the lowest position |
| `last_author` | author id at the highest position |

> Sibling outputs in this project key on `paper_id`, not `work_id`. `work_id` is used here as
> specified; rename the column if it needs to join by name rather than by position.

In [ ]:
import os, sys, time, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa

SRC    = f'{oa.ROOT}/works_au_affs_fixed.csv.gz'
OUT_FP = f'{oa.OUT}/paper_author.parquet'
OLD_TS = f'{oa.OUT}/paper_team_size.parquet'       # the orphaned file this is checked against

# The whole file is read as VARCHAR with the columns named explicitly rather than sniffed: a
# 40.8 GB gzip is sampled from its first rows only, and affiliation_json carries embedded commas
# and quotes. Naming them removes any chance the sniffer shifts a column mid-file.
COLS = {c: 'VARCHAR' for c in [
    'work_id', 'author_position', 'author_position_raw', 'author_position_int', 'author_id',
    'author_display_name', 'raw_author_name', 'is_corresponding', 'affiliation_id',
    'raw_affiliation_string', 'institution_ids', 'countries', 'type', 'orcid',
    'affiliation_json']}

MEM = os.environ.get('NB_DUCKDB_MEM', '180GB')   # a login node cannot do this; see the header
con = duckdb.connect()
con.execute(f"SET memory_limit='{MEM}'")
con.execute(f"SET temp_directory='{oa.CACHE}/duckdb_tmp_authors'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET enable_progress_bar=false")

# One de-duplicated row per (work, author), carrying that author's earliest position.
RAW = f"""
  SELECT replace(work_id,   'https://openalex.org/', '') AS work_id,
         replace(author_id, 'https://openalex.org/', '') AS author_id,
         TRY_CAST(author_position_int AS INTEGER)        AS pos
  FROM read_csv('{SRC}', header=true, columns={COLS})
  WHERE author_id IS NOT NULL AND author_id <> ''
"""
DEDUP = f"SELECT work_id, author_id, min(pos) AS pos FROM ({RAW}) GROUP BY 1, 2"

print(f'source : {SRC}  ({os.path.getsize(SRC)/1e9:.1f} GB gzip)')
print(f'output : {OUT_FP}')
print(f'duckdb : memory_limit {MEM}, temp {oa.CACHE}/duckdb_tmp_authors')

## 1. Build

`string_agg(... ORDER BY pos, author_id)` is what puts `author_list` in author order; the
`author_id` tiebreak makes the output deterministic when two authors somehow share a position.
`arg_min` / `arg_max` pick the ends of that order.

The de-duplication happens **before** the aggregation, so a multi-affiliation author contributes
one entry to `author_list` and one to `team_size`. Sorted by `work_id` on the way out, so a later
join against another output scans a few row groups instead of the file.

In [ ]:
%%time
t0 = time.time()
con.execute(f"""
COPY (
  SELECT work_id,
         string_agg(author_id, ';' ORDER BY pos, author_id) AS author_list,
         count(*)                                           AS team_size,
         arg_min(author_id, pos)                            AS first_author,
         arg_max(author_id, pos)                            AS last_author
  FROM ({DEDUP})
  GROUP BY work_id
  ORDER BY work_id
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)
""")
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB) in {time.time()-t0:.0f}s')
n_works = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'  {n_works:,} works')

## 2. Verification

Four properties, each of which would be a silent defect if it failed.

1. **`author_list` holds no duplicate author.** This is the whole point of the de-duplication
   step — the source file would put a multi-affiliation author in the list two or three times.
2. **`team_size` equals the length of `author_list`.** They are computed by different aggregates
   over the same rows, so agreement is a real check rather than a tautology of the code path.
3. **`first_author` and `last_author` are the ends of `author_list`** — that they came from
   `arg_min`/`arg_max` on the position while the list came from `string_agg ORDER BY` means the
   two orderings have to agree.
4. **Position coverage.** How many works had a null `author_position_int` anywhere; those are the
   only ones whose order rests on the `author_id` tiebreak rather than on the data.

In [ ]:
%%time
chk = con.execute(f"""
WITH s AS (SELECT * FROM read_parquet('{OUT_FP}')),
     x AS (SELECT work_id, author_list, team_size, first_author, last_author,
                  str_split(author_list, ';') AS parts FROM s)
SELECT count(*)                                                          AS works,
       sum(team_size)                                                    AS author_slots,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))   AS dup_in_list,
       count(*) FILTER (WHERE team_size <> len(parts))                   AS size_mismatch,
       count(*) FILTER (WHERE parts[1] <> first_author)                  AS first_mismatch,
       count(*) FILTER (WHERE parts[len(parts)] <> last_author)          AS last_mismatch,
       count(*) FILTER (WHERE team_size = 1)                             AS solo_works
FROM x
""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<16} {int(v):>15,}')
ok = (chk.dup_in_list == 0 and chk.size_mismatch == 0
      and chk.first_mismatch == 0 and chk.last_mismatch == 0)
print(f"\n1-3. {'ALL PASS' if ok else 'FAILED — see the counts above'}")

null_pos = con.execute(f"SELECT count(*) FROM ({RAW}) WHERE pos IS NULL").fetchone()[0]
print(f'4.   rows with a null author_position_int: {null_pos:,}'
      + ('   (order rests on the author_id tiebreak for these)' if null_pos else '   — none'))

display(con.execute(f"""SELECT work_id, team_size, first_author, last_author,
                        length(author_list) AS list_chars
                        FROM read_parquet('{OUT_FP}') ORDER BY team_size DESC LIMIT 5""").fetchdf())
display(con.execute(f"""SELECT team_size, count(*) AS works FROM read_parquet('{OUT_FP}')
                        GROUP BY 1 ORDER BY 1 LIMIT 12""").fetchdf())

## 3. Against the orphaned `paper_team_size.parquet`

`paper_team_size.parquet` has no producer in this project, so this is the only way to find out
what it actually is. Three things are worth separating:

- **coverage** — which works each file has, and what the other one says about the ones it lacks;
- **agreement** — how often the two team sizes are equal on the works both cover;
- **the shape of the disagreement** — a ratio near 1.22 on the mismatches would say the old file
  was built *without* the de-duplication above, which is the specific failure this notebook
  exists to avoid repeating.

In [ ]:
%%time
if not os.path.exists(OLD_TS):
    print(f'{OLD_TS} not present — skipping the comparison')
else:
    cov = con.execute(f"""
    WITH n AS (SELECT work_id, team_size FROM read_parquet('{OUT_FP}')),
         o AS (SELECT paper_id AS work_id, team_size FROM read_parquet('{OLD_TS}'))
    SELECT (SELECT count(*) FROM n)                          AS new_works,
           (SELECT count(*) FROM o)                          AS old_works,
           (SELECT count(*) FROM n JOIN o USING (work_id))    AS both,
           (SELECT count(*) FROM n ANTI JOIN o USING (work_id)) AS new_only,
           (SELECT count(*) FROM o ANTI JOIN n USING (work_id)) AS old_only
    """).fetchdf().iloc[0]
    for k, v in cov.items():
        print(f'  {k:<12} {int(v):>15,}')

    agr = con.execute(f"""
    WITH j AS (SELECT n.work_id, n.team_size AS new_ts, o.team_size AS old_ts
               FROM read_parquet('{OUT_FP}') n
               JOIN (SELECT paper_id AS work_id, team_size FROM read_parquet('{OLD_TS}')) o
                 USING (work_id))
    SELECT count(*)                                              AS n,
           round(100.0*count(*) FILTER (WHERE new_ts = old_ts)/count(*), 3) AS pct_equal,
           round(corr(new_ts, old_ts), 6)                        AS pearson,
           round(corr(ln(new_ts), ln(old_ts)), 6)                AS pearson_log,
           round(avg(new_ts), 4) AS avg_new, round(avg(old_ts), 4) AS avg_old,
           round(avg(old_ts::DOUBLE / new_ts), 4)                AS avg_ratio_old_over_new,
           round(avg(old_ts::DOUBLE / new_ts) FILTER (WHERE new_ts <> old_ts), 4)
                                                                 AS ratio_on_mismatches
    FROM j""").fetchdf().iloc[0]
    print()
    for k, v in agr.items():
        print(f'  {k:<24} {v:>15,}' if isinstance(v, (int,)) else f'  {k:<24} {v:>15}')

    print('\ndifference (old - new), most common:')
    display(con.execute(f"""
    WITH j AS (SELECT n.team_size AS new_ts, o.team_size AS old_ts
               FROM read_parquet('{OUT_FP}') n
               JOIN (SELECT paper_id AS work_id, team_size FROM read_parquet('{OLD_TS}')) o
                 USING (work_id))
    SELECT old_ts - new_ts AS diff, count(*) AS works,
           round(100.0*count(*)/sum(count(*)) OVER (), 3) AS pct
    FROM j GROUP BY 1 ORDER BY works DESC LIMIT 10""").fetchdf())

    print('what the works only one file has look like:')
    display(con.execute(f"""
    SELECT 'new_only' AS side, round(avg(team_size),3) AS avg_team, count(*) AS works
    FROM read_parquet('{OUT_FP}') n
    WHERE NOT EXISTS (SELECT 1 FROM read_parquet('{OLD_TS}') o WHERE o.paper_id = n.work_id)
    UNION ALL
    SELECT 'old_only', round(avg(team_size),3), count(*)
    FROM read_parquet('{OLD_TS}') o
    WHERE NOT EXISTS (SELECT 1 FROM read_parquet('{OUT_FP}') n WHERE n.work_id = o.paper_id)
    """).fetchdf())

## 4. Reading the comparison

- `pct_equal` near 100 with `pearson` ≈ 1 → the old file was built the same way, and this notebook
  restores its reproduction path without changing any number.
- `ratio_on_mismatches` ≈ **1.22** → the old file counted affiliation rows rather than authors,
  and the new column is the corrected one. Anything downstream that used `paper_team_size`
  inherited that inflation.
- Large `old_only` → the old file covers works this source does not reach, and it cannot simply
  be replaced.

Whichever it is, `paper_author.parquet` now has a producer, and `author_list` / `first_author` /
`last_author` are information `paper_team_size` never carried.